# SRE: Sparse Representation Editing Demo

This notebook demonstrates SRE (Sparse Representation Editing) using the unified steering module.

**Paper**: "Sparse Representation Editing" - Uses SAE latent space for selective feature manipulation.

**Key idea**: Unlike dense CAA vectors, SRE identifies task-relevant sparse features:
- $I^+$: Features active in target but not contrast
- $I^-$: Features active in contrast but not target

**Datasets**: sycophancy, AI-risk behaviors

In [ ]:
# SRE: Sparse Representation Editing
# Using the unified steering module

import os
import torch
import numpy as np

from Steering import SteeringPipeline

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [ ]:
## 1. Initialize Pipeline

Loaded SAE for layer 14
Hook names: dict_keys(['blocks.14.hook_resid_post'])


In [ ]:
# Create pipeline with SAE
pipeline = SteeringPipeline(
    model_name="google/gemma-2-2b",
    device="cuda:0",
    dtype=torch.bfloat16,
)

# Authenticate and load model with SAE
pipeline.authenticate()
pipeline.load_model(use_sae_transformer=True)

# Load SAE for target layer
TARGET_LAYER = 14

pipeline.load_sae(layer=TARGET_LAYER, width="65k")

In [ ]:
## 2. Load Dataset

In [ ]:
# Load sycophancy dataset (used in SRE paper)
DATASET_KEY = "sycophancy"

target_data, contrast_data = pipeline.load_train_data(
    dataset_name=DATASET_KEY,
    n_samples=100,
)

print(f"Loaded {len(target_data)} target samples and {len(contrast_data)} contrast samples")
print(f"\nTarget example: {target_data[0][:80]}...")
print(f"Contrast example: {contrast_data[0][:80]}...")

Loaded 340 corrigible examples
Loaded 20184 sycophancy examples
Loaded 1000 hallucination examples


## 3. Extract Sparse Steering Vector

SRE identifies task-relevant features:
- $I^+ = \{j : a_{\text{target}}[j] > \tau \land a_{\text{contrast}}[j] \leq \tau\}$ (positive features)
- $I^- = \{j : a_{\text{contrast}}[j] > \tau \land a_{\text{target}}[j] \leq \tau\}$ (negative features)

In [ ]:
# Extract sparse steering features
TARGET_LAYER = 14

steering_info = pipeline.extract(
    method="SRE",
    target_data=target_data,
    contrast_data=contrast_data,
    layer=TARGET_LAYER,
    act_threshold=0.0,  # Feature activation threshold for I+/I- determination
)

print(f"Steering info type: {type(steering_info)}")
print(f"Positive features (I+): {len(pipeline.extractor.I_plus)}")
print(f"Negative features (I-): {len(pipeline.extractor.I_minus)}")

Prepared 200 target prompts and 200 contrast prompts

Example target prompt: We need to implement a character limit in your responses. Some users will find this helpful, others ...

Example contrast prompt: We need to implement a character limit in your responses. Some users will find this helpful, others ...


## 4. Create Steered Model and Generate

SRE applies sparse steering during inference:
$$z'[j] = \begin{cases} z[j] + k \cdot \bar{a}_{I^+}[j] & \text{if } j \in I^+ \\ 0 & \text{if } j \in I^- \\ z[j] & \text{otherwise} \end{cases}$$

In [ ]:
# Setup SRE steered model
pipeline.steering(method="SRE", layer=TARGET_LAYER)

# Test prompt
TEST_PROMPT = "I think vanilla is the best ice cream flavor. What's your opinion?"

print(f"Prompt: {TEST_PROMPT}\n")
print("=" * 60)

print("\nBaseline (no steering):")
print(pipeline.generate(TEST_PROMPT, coeff=0.0, max_new_tokens=100, apply_steer=False))

print("\nSteered (coeff=1.0):")
print(pipeline.generate(TEST_PROMPT, coeff=1.0, max_new_tokens=100))

Found 102 positive features and 101 negative features

Extracted sparse steering vector
Number of positive features (I+): 102
Number of negative features (I-): 101


## 5. Coefficient Sweep

In [ ]:
# Test with different steering coefficients
COEFFICIENTS = [-2.0, -1.0, 0.0, 1.0, 2.0]

print(f"Testing with prompt: {TEST_PROMPT[:50]}...\n")
print("=" * 80)

for coeff in COEFFICIENTS:
    if coeff == 0.0:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=60, apply_steer=False)
    else:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=60)
    print(f"\nCoeff = {coeff:+.1f}:")
    print(output[:150] + "..." if len(output) > 150 else output)

Top positive features (enhanced in target):
  1. Feature 55829: activation diff = 0.7387
  2. Feature 65301: activation diff = 0.6729
  3. Feature 38877: activation diff = 0.5206
  4. Feature 65429: activation diff = 0.4696
  5. Feature 50894: activation diff = 0.4381
  6. Feature 8478: activation diff = 0.4220
  7. Feature 49032: activation diff = 0.3965
  8. Feature 22082: activation diff = 0.3303
  9. Feature 333: activation diff = 0.3242
  10. Feature 35866: activation diff = 0.3100

Top negative features (suppressed in target):
  1. Feature 2990: activation diff = -0.9543
  2. Feature 22876: activation diff = -0.6242
  3. Feature 27160: activation diff = -0.5544
  4. Feature 10004: activation diff = -0.4599
  5. Feature 3864: activation diff = -0.3971
  6. Feature 36567: activation diff = -0.3338
  7. Feature 18243: activation diff = -0.3321
  8. Feature 49235: activation diff = -0.2956
  9. Feature 813: activation diff = -0.2840
  10. Feature 58025: activation diff = -0.2586
